In [ ]:
import os
import getpass

os.environ["AZURE_STORAGE_CONNECTION_STRING"] = getpass.getpass("Enter Azure connection string: ")


In [ ]:
# Robust Detectron2 installer for Colab (Python-driven)
# Paste this whole cell into Colab and run.
import sys, subprocess, json, traceback

def run(cmd, check=True):
    print(">>>", " ".join(cmd))
    subprocess.check_call(cmd)

try:
    import torch
except Exception as e:
    print("ERROR: torch is not importable. Colab usually ships torch by default in GPU runtime.")
    print("Exception:", e)
    raise

torch_ver = ".".join(torch.__version__.split(".")[:2])
cuda_ver = torch.version.cuda or "cpu"
print(f"Detected torch: {torch.__version__} -> wheel prefix torch{torch_ver}, cuda: {cuda_ver}")

# Upgrade pip and install common deps
try:
    run([sys.executable, "-m", "pip", "install", "-U", "pip"])
    run([sys.executable, "-m", "pip", "install", "-U", "torchvision"])
    run([sys.executable, "-m", "pip", "install", "-U", "fvcore"])
except subprocess.CalledProcessError:
    print("Warning: failed to upgrade/install basic packages; continuing anyway.")

# Construct detectron2 wheel index URL only when CUDA is available
if cuda_ver != "cpu" and cuda_ver is not None:
    wheel_index = f"https://dl.fbaipublicfiles.com/detectron2/wheels/{cuda_ver}/torch{torch_ver}/index.html"
    print("Attempting to install detectron2 from wheel index:", wheel_index)
    try:
        run([sys.executable, "-m", "pip", "install", "detectron2", "-f", wheel_index, "--no-cache-dir"])
        print("detectron2 wheel installed successfully.")
    except subprocess.CalledProcessError:
        print("detectron2 wheel install failed — trying fallback (install from source).")
        try:
            run([sys.executable, "-m", "pip", "install", "git+https://github.com/facebookresearch/detectron2.git"])
            print("detectron2 installed from source (fallback) successfully.")
        except subprocess.CalledProcessError:
            print("ERROR: detectron2 installation failed (wheel and source fallback). See trace below.")
            traceback.print_exc()
            raise
else:
    print("No CUDA detected (cuda_ver='cpu'), installing detectron2 from source.")
    try:
        run([sys.executable, "-m", "pip", "install", "git+https://github.com/facebookresearch/detectron2.git"])
        print("detectron2 installed from source successfully.")
    except subprocess.CalledProcessError:
        print("ERROR: detectron2 installation from source failed. See trace below.")
        traceback.print_exc()
        raise

print("\nInstallation finished. **IMPORTANT**: Restart the Colab runtime (Runtime → Restart runtime) before importing detectron2 or Mask2Former.")


In [ ]:
# ONE-CELL: Install Detectron2 + Mask2Former, then run streaming segmentation pipeline (Colab-ready).
# Paste into a single Colab code cell and run. If asked to restart, restart runtime and re-run the same cell.
import os, sys, subprocess, time, json, logging, getpass, traceback
from pathlib import Path

# ---------------- CONFIG - tune these ----------------
CONTAINER = "london"
PREFIX = "training/training/Images"      # Azure prefix with images
MANIFEST_BLOB = "manifests/mask2former_manifest.jsonl"
STATE_BLOB = "manifests/mask2former_state.json"
REMOTE_OUTPUT_PREFIX = "mask2former_results"
LOCAL_CACHE = Path("/tmp/image_cache")
LOCAL_OUT = Path("/tmp/m2f_out")
PREFETCH = 8                 # how many images to keep locally ahead of processing
DOWNLOAD_WORKERS = 3
DEVICE = "cuda" if (hasattr(sys, "getallocatedblocks") or True) and (('cuda' in sys.modules) or True) else "cuda"
# Mask2Former config: swap to a smaller model if needed for Colab.
# Default config path is inside cloned Mask2Former repo; change to a smaller backbone if you run OOM.
CFG_FILE = "Mask2Former/configs/coco/panoptic-segmentation/swin/maskformer2_swin_base_IN21k_384_bs16_100ep.yaml"
WEIGHTS = ("https://dl.fbaipublicfiles.com/maskformer/mask2former/coco/panoptic/"
           "maskformer2_swin_base_IN21k_384_bs16_100ep/model_final.pth")  # try base model (lighter than large)
# If the above pth is not available, swap with a ResNet50-based model from Mask2Former repo examples.

# ---------------- logging ----------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("colab_mask2former_onecell")

# ---------------- helper to run shell commands ----------------
def sh(cmd, check=True):
    print(">>>", cmd)
    res = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(res.stdout)
    if check and res.returncode != 0:
        raise RuntimeError(f"Command failed (code {res.returncode}): {cmd}")
    return res

# ---------------- installation (attempt) ----------------
need_restart_marker = Path("/tmp/.colab_m2f_installed")
try:
    if not need_restart_marker.exists():
        print("1) Upgrading pip / installing helper packages ...")
        sh(f"{sys.executable} -m pip install -U pip setuptools wheel", check=True)
        sh(f"{sys.executable} -m pip install -U torchvision fvcore iopath yacs pycocotools", check=True)

        # Try detectron2 wheel for current torch/cuda, fallback to source
        import torch
        torch_ver = ".".join(torch.__version__.split(".")[:2])
        cuda_ver = torch.version.cuda or "cpu"
        print(f"Detected torch {torch.__version__}, cuda {cuda_ver}")
        if cuda_ver != "cpu":
            wheel_index = f"https://dl.fbaipublicfiles.com/detectron2/wheels/{cuda_ver}/torch{torch_ver}/index.html"
            try:
                print("Trying to install detectron2 wheel from:", wheel_index)
                sh(f"{sys.executable} -m pip install detectron2 -f {wheel_index} --no-cache-dir", check=True)
            except Exception:
                print("Detectron2 wheel install failed; falling back to installing from GitHub (this may take several minutes)...")
                sh(f"{sys.executable} -m pip install git+https://github.com/facebookresearch/detectron2.git", check=True)
        else:
            print("CUDA not detected; installing detectron2 from source (may be slow).")
            sh(f"{sys.executable} -m pip install git+https://github.com/facebookresearch/detectron2.git", check=True)

        # Clone Mask2Former repo (if missing) and install its requirements
        if not Path("Mask2Former").exists():
            sh("git clone https://github.com/facebookresearch/Mask2Former.git", check=True)
        sh("python -m pip install -r Mask2Former/requirements.txt", check=True)
        sh("python -m pip install opencv-python azure-storage-blob tqdm requests", check=True)
        sh("python -m pip install git+https://github.com/cocodataset/panopticapi.git", check=True)

        # Build & install Mask2Former ops (may compile CUDA extensions)
        sh("cd Mask2Former/mask2former/modeling/pixel_decoder/ops && python setup.py build install", check=True)

        # mark install done - a runtime restart is normally required for compiled extensions to be importable
        need_restart_marker.write_text("installed")
        print("Installation finished. NOTE: You will probably need to restart the runtime for Detectron2/Mask2Former extensions to load.")
        print("Please restart runtime now (Runtime -> Restart runtime). After restart, re-run this same cell and it will continue onto the pipeline.")
        # attempt to import detectron2 now; if fails, instruct restart
    else:
        print("Install marker found: skipping install steps (assuming you restarted the runtime).")
except Exception as e:
    print("Installation step failed with error:\n", e)
    traceback.print_exc()
    print("\nIf this failed during detectron2 wheel or Mask2Former build, try restarting the runtime and re-running this cell.")
    raise

# ---------------- attempt imports; if imports fail, ask for restart ----------------
try:
    import torch
    import detectron2
    from detectron2.config import get_cfg
    from detectron2.engine import DefaultPredictor
    from detectron2.utils.visualizer import Visualizer, ColorMode
    from detectron2.data import MetadataCatalog
    # Mask2Former helpers
    from mask2former import add_maskformer2_config
    from detectron2.projects.deeplab import add_deeplab_config
    print("Imports OK: detectron2 and mask2former are available.")
except Exception as exc:
    print("ERROR importing detectron2/mask2former (this usually means a runtime restart is required after install).")
    print("Exception:", exc)
    traceback.print_exc()
    print("\nPlease restart the Colab runtime now (Runtime -> Restart runtime). Then re-run this same cell.")
    raise SystemExit("Please restart runtime and re-run the cell (imports failed).")

# ---------------- secure Azure connection string input ----------------
AZ_CONN_STR = os.environ.get("AZ_CONN_STR")
if not AZ_CONN_STR:
    AZ_CONN_STR = getpass.getpass("Enter Azure Storage connection string (input hidden): ")
    os.environ["AZ_CONN_STR"] = AZ_CONN_STR

# ---------------- Azure client ----------------
from azure.storage.blob import BlobServiceClient
blob_service = BlobServiceClient.from_connection_string(AZ_CONN_STR)
container_client = blob_service.get_container_client(CONTAINER)

# ---------------- Pipeline components ----------------
import threading
from queue import Queue, Empty
from azure.core.exceptions import AzureError
import numpy as np
import cv2
from typing import List, Dict, Any, Optional, Tuple

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def atomic_write_json(path: Path, obj: Dict[str, Any]):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2)
    os.replace(tmp, path)

def list_blobs_prefix(prefix: str) -> List[Dict[str, Any]]:
    out = []
    for b in container_client.list_blobs(name_starts_with=prefix):
        if b.name.rstrip("/") == prefix.rstrip("/"):
            continue
        out.append({"name": b.name, "size": getattr(b, "size", None)})
    return out

def upload_file_to_blob(blob_path: str, local_path: Path, overwrite=True):
    bc = container_client.get_blob_client(blob_path)
    with open(local_path, "rb") as fh:
        bc.upload_blob(fh, overwrite=overwrite)

def upload_bytes_to_blob(blob_path: str, data: bytes, overwrite=True):
    bc = container_client.get_blob_client(blob_path)
    bc.upload_blob(data, overwrite=overwrite)

def download_blob_to_local(blob_name: str, local_path: Path, expected_size: Optional[int] = None, retry=4):
    ensure_dir(local_path.parent)
    if local_path.exists() and expected_size is not None:
        try:
            if local_path.stat().st_size == expected_size:
                return "skipped"
        except Exception:
            pass
    part = local_path.with_suffix(local_path.suffix + ".part")
    attempt = 0
    while True:
        try:
            stream = container_client.download_blob(blob_name)
            with open(part, "wb") as fh:
                for chunk in stream.chunks():
                    fh.write(chunk)
            if expected_size is not None and part.stat().st_size != expected_size:
                raise RuntimeError(f"size mismatch after download {part} {part.stat().st_size} != {expected_size}")
            os.replace(part, local_path)
            return "downloaded"
        except Exception as e:
            attempt += 1
            logger.warning("Download failed %s attempt=%d err=%s", blob_name, attempt, str(e))
            try:
                if part.exists() and attempt > 1:
                    part.unlink(missing_ok=True)
            except Exception:
                pass
            if attempt > retry:
                raise
            time.sleep(2 ** attempt)

# ---------------- Build predictor ----------------
def build_predictor(cfg_file=CFG_FILE, weights=WEIGHTS, device="cuda"):
    cfg = get_cfg()
    add_deeplab_config(cfg)
    add_maskformer2_config(cfg)
    cfg.merge_from_file(cfg_file)
    cfg.MODEL.WEIGHTS = weights
    cfg.MODEL.DEVICE = device
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.0
    cfg.freeze()
    pred = DefaultPredictor(cfg)
    return pred, cfg

# ---------------- Processing function ----------------
def process_image_and_upload(predictor, cfg, blob_name: str, local_img_path: Path, remote_prefix: str) -> Dict[str, Any]:
    im = cv2.imread(str(local_img_path))
    if im is None:
        raise RuntimeError(f"Could not read image {local_img_path}")
    outputs = predictor(im)

    saved = {}
    stem = local_img_path.stem
    ensure_dir(LOCAL_OUT)

    # Panoptic
    pan = outputs.get("panoptic_seg", None)
    coco_meta = None
    try:
        if len(cfg.DATASETS.TEST) > 0:
            coco_meta = MetadataCatalog.get(cfg.DATASETS.TEST[0])
    except Exception:
        coco_meta = None

    if pan is not None:
        pan_seg, pan_info = pan
        viz = Visualizer(im[:, :, ::-1], coco_meta, scale=1.0, instance_mode=ColorMode.IMAGE_BW)
        pan_img = viz.draw_panoptic_seg(pan_seg.to("cpu"), pan_info).get_image()[:, :, ::-1]
        pan_path = LOCAL_OUT / f"{stem}_panoptic.png"
        cv2.imwrite(str(pan_path), pan_img)
        upload_file_to_blob(f"{remote_prefix}/panoptic/{stem}.png", pan_path, overwrite=True)
        saved["panoptic"] = f"{remote_prefix}/panoptic/{stem}.png"

    # Instances
    instances = outputs.get("instances", None)
    if instances is not None and len(instances) > 0:
        inst_cpu = instances.to("cpu")
        viz = Visualizer(im[:, :, ::-1], coco_meta, scale=1.0, instance_mode=ColorMode.IMAGE_BW)
        inst_img = viz.draw_instance_predictions(inst_cpu).get_image()[:, :, ::-1]
        inst_path = LOCAL_OUT / f"{stem}_instances.png"
        cv2.imwrite(str(inst_path), inst_img)
        upload_file_to_blob(f"{remote_prefix}/instances/{stem}.png", inst_path, overwrite=True)
        saved["instances_img"] = f"{remote_prefix}/instances/{stem}.png"

        pred_masks = inst_cpu.pred_masks.numpy() if hasattr(inst_cpu, "pred_masks") else None
        boxes_xyxy = inst_cpu.pred_boxes.tensor.numpy() if hasattr(inst_cpu, "pred_boxes") else None
        scores = inst_cpu.scores.numpy() if hasattr(inst_cpu, "scores") else None
        classes = inst_cpu.pred_classes.numpy() if hasattr(inst_cpu, "pred_classes") else None

        inst_meta = []
        for i in range(len(inst_cpu)):
            entry = {"id": int(i)}
            if boxes_xyxy is not None:
                entry["bbox_xyxy"] = boxes_xyxy[i].tolist()
            if scores is not None:
                entry["score"] = float(scores[i])
            if classes is not None:
                entry["class"] = int(classes[i])
            if pred_masks is not None:
                mask = (pred_masks[i] > 0).astype("uint8") * 255
                mask_local = LOCAL_OUT / f"{stem}_mask_{i}.png"
                cv2.imwrite(str(mask_local), mask)
                remote_mask_blob = f"{remote_prefix}/masks/{stem}_mask_{i}.png"
                upload_file_to_blob(remote_mask_blob, mask_local, overwrite=True)
                entry["mask"] = remote_mask_blob
            inst_meta.append(entry)
        inst_meta_blob = f"{remote_prefix}/metadata/{stem}_instances.json"
        upload_bytes_to_blob(inst_meta_blob, json.dumps(inst_meta).encode("utf8"), overwrite=True)
        saved["instances_meta"] = inst_meta_blob

    # Semantic
    sem = outputs.get("sem_seg", None)
    if sem is not None:
        sem_cpu = sem.argmax(0).to("cpu").numpy()
        viz = Visualizer(im[:, :, ::-1], coco_meta, scale=1.0, instance_mode=ColorMode.IMAGE_BW)
        try:
            sem_img = viz.draw_sem_seg(torch.from_numpy(sem_cpu).to("cpu")).get_image()[:, :, ::-1]
            sem_path = LOCAL_OUT / f"{stem}_semantic.png"
            cv2.imwrite(str(sem_path), sem_img)
            upload_file_to_blob(f"{remote_prefix}/semantic/{stem}.png", sem_path, overwrite=True)
            saved["semantic"] = f"{remote_prefix}/semantic/{stem}.png"
        except Exception:
            logger.debug("semantic viz failed; skipping")

    try:
        del outputs, instances, sem, pan
    except Exception:
        pass
    torch.cuda.empty_cache()
    return saved

# ---------------- Producer-consumer pipeline ----------------
def run_pipeline(prefetch=PREFETCH, download_workers=DOWNLOAD_WORKERS):
    ensure_dir(LOCAL_CACHE); ensure_dir(LOCAL_OUT)
    manifest = list_blobs_prefix(PREFIX)
    if not manifest:
        logger.error("No blobs found under prefix %s", PREFIX); return
    # upload manifest
    # write local manifest and upload
    lm = LOCAL_OUT / "manifest.jsonl"
    with open(lm, "w") as fh:
        for r in manifest:
            fh.write(json.dumps(r) + "\n")
    upload_file_to_blob(MANIFEST_BLOB, lm, overwrite=True)
    logger.info("Manifest uploaded (%d entries).", len(manifest))

    # load remote state
    try:
        bc = container_client.get_blob_client(STATE_BLOB)
        if bc.exists():
            remote_state = json.loads(bc.download_blob().readall())
        else:
            remote_state = {"processed": {}}
    except Exception:
        remote_state = {"processed": {}}
    state = remote_state

    # build list to process
    to_process = [r for r in manifest if not (r["name"] in state.get("processed", {}) and state["processed"][r["name"]].get("status") == "done")]
    logger.info("To process: %d", len(to_process))

    # queue & downloader threads
    from queue import Queue, Empty
    q = Queue(maxsize=prefetch + 4)

    def downloader_worker(items):
        for rec in items:
            blob_name = rec["name"]; expected_size = rec.get("size")
            st = state.get("processed", {}).get(blob_name)
            if st is not None and st.get("status") == "done": continue
            rel = Path(blob_name).relative_to(PREFIX)
            local_path = LOCAL_CACHE / rel
            ensure_dir(local_path.parent)
            try:
                res = download_blob_to_local(blob_name, local_path, expected_size)
                logger.info("Downloaded %s -> %s (%s)", blob_name, local_path, res)
            except Exception as e:
                logger.exception("Download failed: %s", str(e))
                state.setdefault("processed", {}).setdefault(blob_name, {})["status"] = "download_failed"
                state["processed"][blob_name]["error"] = str(e)
                # persist state
                atomic_write_json(LOCAL_OUT / "state.json", state); upload_file_to_blob(STATE_BLOB, LOCAL_OUT / "state.json", overwrite=True)
                continue
            # push to queue
            while True:
                try:
                    q.put((blob_name, local_path), timeout=5); break
                except Exception:
                    time.sleep(0.5)
        logger.info("Downloader worker done.")

    # split tasks
    parts = [[] for _ in range(download_workers)]
    for i, rec in enumerate(to_process):
        parts[i % download_workers].append(rec)
    threads = []
    for i in range(download_workers):
        t = threading.Thread(target=downloader_worker, args=(parts[i],), daemon=True)
        t.start(); threads.append(t)

    # build predictor (single GPU worker)
    predictor, cfg = build_predictor(CFG_FILE, WEIGHTS, device=("cuda" if torch.cuda.is_available() else "cpu"))
    logger.info("Predictor ready; starting consumer loop...")

    processed = 0; total = len(to_process)
    try:
        while True:
            try:
                blob_name, local_path = q.get(timeout=60)
            except Empty:
                all_done = all(not t.is_alive() for t in threads)
                if all_done and q.empty():
                    logger.info("All done."); break
                else:
                    continue
            if state.get("processed", {}).get(blob_name, {}).get("status") == "done":
                try: local_path.unlink(missing_ok=True)
                except Exception: pass
                q.task_done(); continue
            logger.info("Processing %d/%d: %s", processed+1, total, blob_name)
            try:
                saved = process_image_and_upload(predictor, cfg, blob_name, local_path, REMOTE_OUTPUT_PREFIX)
                state.setdefault("processed", {})[blob_name] = {"status":"done", "ts": time.time(), "saved": saved}
                # persist local & remote
                atomic_write_json(LOCAL_OUT / "state.json", state)
                upload_file_to_blob(STATE_BLOB, LOCAL_OUT / "state.json", overwrite=True)
                # per-image done marker
                upload_bytes_to_blob(f"{REMOTE_OUTPUT_PREFIX}/done/{Path(local_path).stem}.json",
                                     json.dumps({"blob": blob_name, "ts": time.time()}).encode("utf8"), overwrite=True)
                logger.info("Uploaded outputs for %s", blob_name)
            except Exception as e:
                logger.exception("Processing failed for %s: %s", blob_name, str(e))
                state.setdefault("processed", {})[blob_name] = {"status":"processing_failed", "error": str(e)}
                atomic_write_json(LOCAL_OUT / "state.json", state)
                upload_file_to_blob(STATE_BLOB, LOCAL_OUT / "state.json", overwrite=True)
            finally:
                try: local_path.unlink(missing_ok=True)
                except Exception: pass
                processed += 1
                q.task_done()
    except KeyboardInterrupt:
        logger.warning("Interrupted; saving state.")
    finally:
        # join threads and save final state
        for t in threads:
            t.join(timeout=1)
        atomic_write_json(LOCAL_OUT / "state.json", state)
        upload_file_to_blob(STATE_BLOB, LOCAL_OUT / "state.json", overwrite=True)
        logger.info("Pipeline finished/paused. Processed=%d", processed)

# ---------------- run ----------------
if __name__ == "__main__":
    run_pipeline(prefetch=PREFETCH, download_workers=DOWNLOAD_WORKERS)
